# CLV 이중축 M2 다중-seed 식별 대조군
기존 seed 42·43·44의 M1, encoder, `dual_clv_fixed` 결과를 재사용하고 seed 43·44에서 `dual_shuffled_user`, `dual_adapter_only`만 학습합니다. Dunnhumby 전체기간과 H&M 60일 validation을 순차 실행하며 test·holdout·H&M 2년·lambda 재탐색은 하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess
REVIEWED_SHA = '3bb3fe43dc75d66782ff35ab0bc7c29161294b59'
REPO_DIR = '/content/clv-m2-lightgcn-runner'
os.chdir('/content')
shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA
print('코드 준비 완료:', actual_sha)

In [ ]:
from pathlib import Path
from IPython.display import display
import torch
from lightgcn_clv_dual_multiseed_controls import configure_multiseed_controls, run_multiseed_controls

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
ROOT = Path('/content/drive/MyDrive/논문/data')
DUN = ROOT / 'results_clv_dual_dunnhumby'
HM = ROOT / 'results_clv_dual_hm_w60'

configs = {
    'dunnhumby': configure_multiseed_controls(
        'dunnhumby',
        DUN / 'clv_dual_dunnhumby_662adb04aa.json',
        DUN / 'multiseed_validation/clv_dual_multiseed_dunnhumby.json',
        out_dir=DUN / 'multiseed_control_validation',
    ),
    'hm_w60': configure_multiseed_controls(
        'hm',
        HM / 'clv_dual_hm_fdb66ee106.json',
        HM / 'multiseed_validation/clv_dual_multiseed_hm.json',
        short_hm=True,
        out_dir=HM / 'multiseed_control_validation',
    ),
}
for name, cfg in configs.items(): print(name, cfg)

## 실행
아래 셀 하나가 두 데이터셋을 순차 실행합니다. 새로 학습하는 것은 seed 43·44의 두 대조군 adapter뿐입니다.

In [ ]:
results = {}
for name in ('dunnhumby', 'hm_w60'):
    print(f'\n===== {name}: 식별 대조군 시작 =====')
    results[name] = run_multiseed_controls(configs[name])
    print(f'===== {name}: 완료 =====')

In [ ]:
for name, frame in results.items():
    print(f'\n===== {name}: 선택점 비교 =====')
    display(frame[['seed', 'model_id', 'gate_shape', 'lambda', 'recall@10', 'ndcg@10', 'revenue@10', 'arp@10', 'coverage@10', 'n_distinct@10', 'eff_catalog@10']])
    decision = frame.attrs['control_reproducibility_decision']
    print('대조군 식별 통과:', decision['success'])
    print('통과하지 못한 대조군:', decision['failed_controls'])
    print('대조군별 비교:', decision['comparisons'])
    print('결과 파일:', frame.attrs['result_paths'])
print('완료: 결과를 공유하기 전에는 다음 실험을 실행하지 않습니다.')